# GDELT Extraction — extending to 2024

Same pattern as `GDELT_Extraction_full.ipynb`: the same GDELT 1.0 daily-file puller, the same
`filter_events`/`extract_events_for_range`/`pull_gdelt_years` functions, and the same
`stream_score`/`_finalize_node`/`_finalize_pair` scoring pipeline -- just re-pointed at 2024 and
then merged with the existing 2017-2023 outputs rather than replacing them.

**Data-availability note, carried over honestly from the original notebook's own reasoning:**
GDELT 1.0's daily-file feed only exists from **2015-02-19** onward. 2024 is well inside that
range, so this extension is straightforward. (This note matters more for a *future* extension
back to, say, 2000 -- that's a genuinely different, harder pull; see the caveat in the chat
reply this notebook was generated alongside.)

## Config -- only this cell should need editing

In [1]:
import os, io, zipfile, time
import pandas as pd
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

# ---- paths match tree_repo.txt: interim/ for outputs, cache/ for checkpoints ----
INTERIM_DIR = os.path.join("..", "..", "data", "interim")
CACHE_DIR = os.path.join("..", "..", "data", "cache")

START_YEAR, END_YEAR = 2024, 2024

COUNTRIES = None          # None = every country, matching the original full-dataset pull
EVENT_ROOT_CODES = None   # None = every event type -- CRIT filtering happens downstream, same as before

MIN_MENTIONS = 5
MIN_SOURCES = 2

CHECKPOINT_DIR = os.path.join(CACHE_DIR, "gdelt_chunks_2024")
OUTPUT_PATH_2024 = os.path.join(INTERIM_DIR, "gdelt_events_2024.parquet")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [2]:
# GDELT 1.0 daily files = 58 columns -- identical schema to the 2017-2023 pull
GDELT_COLUMNS_V1 = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode",
    "Actor1EthnicCode","Actor1Religion1Code","Actor1Religion2Code",
    "Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode",
    "Actor2EthnicCode","Actor2Religion1Code","Actor2Religion2Code",
    "Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code","Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code","Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode",
    "ActionGeo_ADM1Code","ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL",
]

KEEP_COLS = [
    "GLOBALEVENTID","SQLDATE","DATEADDED","Actor1Name","Actor1CountryCode",
    "Actor2Name","Actor2CountryCode","ActionGeo_CountryCode","ActionGeo_FullName",
    "EventCode","EventBaseCode","EventRootCode",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone","SOURCEURL",
]

## Date helpers, downloader, filter -- unchanged from the original

In [3]:
def generate_gdelt_dates(start_date, end_date):
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    days = []
    while cur <= end:
        days.append(cur.strftime("%Y%m%d")); cur += timedelta(days=1)
    return days

def month_ranges(y0, y1):
    out, cur, end = [], datetime(y0,1,1), datetime(y1,12,31)
    while cur <= end:
        nxt = datetime(cur.year + (cur.month==12), (cur.month % 12)+1, 1)
        out.append((cur.strftime("%Y-%m-%d"), (nxt-timedelta(days=1)).strftime("%Y-%m-%d")))
        cur = nxt
    return out

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (research data pull)"})

def download_gdelt_file_v1(date_str, retries=4, pause=0.4):
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=60)
            if r.status_code == 200:
                z = zipfile.ZipFile(io.BytesIO(r.content))
                raw = pd.read_csv(z.open(z.namelist()[0]), sep="\t",
                                  header=None, dtype=str, low_memory=False)
                if raw.shape[1] != len(GDELT_COLUMNS_V1):
                    print(f"  \u26a0 {date_str}: {raw.shape[1]} cols (expected {len(GDELT_COLUMNS_V1)})")
                    return None
                raw.columns = GDELT_COLUMNS_V1
                time.sleep(pause)
                return raw
            if r.status_code == 404:
                return None
            print(f"  {date_str}: HTTP {r.status_code} (throttled?), retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
        except Exception as e:
            print(f"  {date_str}: {type(e).__name__}, retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
    print(f"  {date_str}: gave up after {retries} tries")
    return None

def filter_events(df, countries=COUNTRIES, event_root_codes=EVENT_ROOT_CODES,
                   min_mentions=MIN_MENTIONS, min_sources=MIN_SOURCES):
    df = df.copy()
    for c in ["NumMentions","NumSources","NumArticles","GoldsteinScale","AvgTone"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if countries is not None:
        df = df[
            df["Actor1CountryCode"].isin(countries) |
            df["Actor2CountryCode"].isin(countries) |
            df["ActionGeo_CountryCode"].isin(countries)
        ]
    if df.empty:
        return pd.DataFrame()

    if event_root_codes is not None:
        df = df[df["EventRootCode"].isin(event_root_codes)]
    if df.empty:
        return pd.DataFrame()

    df = df[(df["NumMentions"] >= min_mentions) & (df["NumSources"] >= min_sources)]
    if df.empty:
        return pd.DataFrame()

    return df[[c for c in KEEP_COLS if c in df.columns]]

## Pull functions -- unchanged, just parameterized to 2024

In [4]:
def extract_events_for_range(start_date, end_date, output_path):
    all_events = []
    for d in tqdm(generate_gdelt_dates(start_date, end_date)):
        df = download_gdelt_file_v1(d)
        if df is None or df.empty:
            continue
        filt = filter_events(df)
        if not filt.empty:
            all_events.append(filt)
    if not all_events:
        print("No matching events found."); return pd.DataFrame()
    result = pd.concat(all_events, ignore_index=True)
    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])
    result.to_parquet(output_path, index=False)
    print(f"Saved {len(result):,} events to {output_path}")
    return result

def pull_gdelt_years(y0=START_YEAR, y1=END_YEAR):
    chunk_files = []
    for start, end in month_ranges(y0, y1):
        tag = start[:7]
        out = f"{CHECKPOINT_DIR}/gdelt_{tag}.parquet"
        chunk_files.append(out)
        if os.path.exists(out):
            print(f"\u2713 {tag} done \u2014 skipping"); continue
        print(f"\n=== {tag} ===")
        extract_events_for_range(start, end, output_path=out)
    parts = [pd.read_parquet(f) for f in chunk_files if os.path.exists(f)]
    events = pd.concat(parts, ignore_index=True).drop_duplicates("GLOBALEVENTID")
    events.to_parquet(OUTPUT_PATH_2024, index=False)
    print(f"\nFINAL: {len(events):,} events -> {OUTPUT_PATH_2024}")
    return events

## Test on one week before committing to the full year

In [5]:
test = extract_events_for_range("2024-01-01", "2024-01-07", output_path="test_events_2024.parquet")
test.shape

100%|██████████| 7/7 [00:15<00:00,  2.28s/it]


Saved 134,785 events to test_events_2024.parquet


(134785, 19)

## Full 2024 pull -- run once the test above looks right

In [6]:
events_2024 = pull_gdelt_years(START_YEAR, END_YEAR)

✓ 2024-01 done — skipping
✓ 2024-02 done — skipping
✓ 2024-03 done — skipping
✓ 2024-04 done — skipping
✓ 2024-05 done — skipping
✓ 2024-06 done — skipping
✓ 2024-07 done — skipping
✓ 2024-08 done — skipping
✓ 2024-09 done — skipping
✓ 2024-10 done — skipping
✓ 2024-11 done — skipping
✓ 2024-12 done — skipping

FINAL: 8,139,847 events -> ../../data/interim/gdelt_events_2024.parquet


## Merge with the existing 2017-2023 events file

Same dedup-by-`GLOBALEVENTID` logic as the original notebook's chunk-combining cell.

In [7]:
import pyarrow as pa
import pyarrow.parquet as pq

EXISTING_PATH = os.path.join(INTERIM_DIR, "gdelt_events_2017_2023.parquet")
NEW_PATH = os.path.join(INTERIM_DIR, "gdelt_events_2024.parquet")
MERGED_EVENTS_PATH = os.path.join(INTERIM_DIR, "gdelt_events_2017_2024.parquet")

seen_ids = set()
writer = None
total_in, total_written = 0, 0

for src_path, label in [(EXISTING_PATH, "2017-2023"), (NEW_PATH, "2024")]:
    pf = pq.ParquetFile(src_path)
    for b in range(pf.num_row_groups):
        chunk = pf.read_row_group(b).to_pandas()
        total_in += len(chunk)

        # dedupe against everything written so far (cheap: just an ID set, not full rows)
        mask = ~chunk["GLOBALEVENTID"].isin(seen_ids)
        chunk = chunk[mask]
        seen_ids.update(chunk["GLOBALEVENTID"].tolist())

        if chunk.empty:
            continue
        total_written += len(chunk)
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(MERGED_EVENTS_PATH, table.schema)
        writer.write_table(table)

        if b % 20 == 0:
            print(f"  {label} row-group {b}/{pf.num_row_groups} | written so far: {total_written:,}")

if writer:
    writer.close()

print(f"\nDONE -> {MERGED_EVENTS_PATH}")
print(f"scanned {total_in:,} rows, wrote {total_written:,} deduped rows")

combined_check = pd.read_parquet(MERGED_EVENTS_PATH, columns=["date"])
print("years:", sorted(pd.to_datetime(combined_check['date']).dt.year.unique()))

  2017-2023 row-group 0/93 | written so far: 1,048,576
  2017-2023 row-group 20/93 | written so far: 15,424,418
  2017-2023 row-group 40/93 | written so far: 32,165,205


OSError: [Errno 28] Error writing bytes to file. Detail: [errno 28] No space left on device

In [5]:
import pyarrow.parquet as pq
import glob

def row_count(path):
    return pq.ParquetFile(path).metadata.num_rows   # reads only the footer -- no data loaded, KB not GB

final_rows = row_count("../../data/interim/gdelt_events_2017_2023.parquet")
print("final file rows:", final_rows)

for chunk_dir in ["../../data/cache/gdelt_chunks_full", "../../data/cache/gdelt_chunks_raw"]:
    files = glob.glob(f"{chunk_dir}/*.parquet")
    total = sum(row_count(f) for f in files)
    print(chunk_dir, "->", total, f"rows across {len(files)} files")

final file rows: 67487378
../../data/cache/gdelt_chunks_full -> 67487378 rows across 84 files
../../data/cache/gdelt_chunks_raw -> 86220668 rows across 84 files


In [6]:
import pandas as pd
sample = pd.read_parquet("../../data/cache/gdelt_chunks_raw/raw_2017-01.parquet", columns=["EventRootCode", "NumMentions", "NumSources"])
print(sample.shape)
print("root codes present:", sorted(sample["EventRootCode"].astype(str).unique())[:20])
print("min mentions:", sample["NumMentions"].min(), "min sources:", sample["NumSources"].min())

(1520683, 3)
root codes present: ['--', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']
min mentions: 3 min sources: 1


In [4]:
import pyarrow.parquet as pq
import glob

def row_count(path):
    return pq.ParquetFile(path).metadata.num_rows   # reads only the footer -- no data loaded, KB not GB

final_rows = row_count("../../data/interim/gdelt_events_2017_2024.parquet")
print("final file rows:", final_rows)

for chunk_dir in ["data/cache/gdelt_chunks_full", "data/cache/gdelt_chunks_raw"]:
    files = glob.glob(f"{chunk_dir}/*.parquet")
    total = sum(row_count(f) for f in files)
    print(chunk_dir, "->", total, f"rows across {len(files)} files")

ArrowInvalid: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

## Re-run the scoring/aggregation pipeline over the combined file

Identical logic to `GDELT_Extraction_full.ipynb` cells 19-23 (`stream_score`, `_finalize_node`,
`_finalize_pair`) -- just re-run over `gdelt_events_2017_2024.parquet` instead of the
2017-2023-only file, and the year filter widened to include 2024.

In [9]:
import sys
sys.path.append("../../src/utils")
import pyarrow.parquet as pq
import numpy as np
import pandas as pd
import os
from country_codes import ISO3_2_M49

INTERIM_DIR = os.path.join("..", "..", "data", "interim")

COLS = ["GLOBALEVENTID","Actor1CountryCode","Actor2CountryCode",
        "EventRootCode","GoldsteinScale","NumMentions","AvgTone","date"]
CRIT = ["14","17","18","19","20"]
K = 10
iso2m49 = {**ISO3_2_M49, "TWN": "490"}

def stream_score(path=os.path.join(INTERIM_DIR, "gdelt_events_2024.parquet")):
    pf = pq.ParquetFile(path)
    node_acc, pair_acc = {}, {}
    node_topk, pair_topk = {}, {}

    def add(acc, key, score, gold, ment, tone):
        s = acc.get(key)
        if s is None:
            acc[key] = [1, score, score, gold*max(ment,1), max(ment,1), tone]
        else:
            s[0]+=1; s[1]+=score; s[2]=max(s[2],score); s[3]+=gold*max(ment,1); s[4]+=max(ment,1); s[5]+=tone

    def add_topk(acc, key, score, gold, ment, tone):
        buf = acc.setdefault(key, [])
        buf.append((score, gold, ment, tone))
        if len(buf) > 200:
            buf.sort(key=lambda x:-x[0]); del buf[100:]

    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b, columns=COLS).to_pandas()
        ch = ch.dropna(subset=["Actor1CountryCode","date"])
        ch["EventRootCode"] = ch["EventRootCode"].astype(str)
        ch = ch[ch["EventRootCode"].isin(CRIT)]
        if ch.empty: continue
        ch["ym"]   = pd.to_datetime(ch["date"],errors="coerce").dt.to_period("M")
        ch = ch.dropna(subset=["ym"])
        g = pd.to_numeric(ch["GoldsteinScale"],errors="coerce").fillna(0).values
        m = pd.to_numeric(ch["NumMentions"],errors="coerce").fillna(0).values
        t = pd.to_numeric(ch["AvgTone"],errors="coerce").fillna(0).values
        sc = np.abs(g)*m
        a1 = ch["Actor1CountryCode"].values; a2 = ch["Actor2CountryCode"].values
        ym = ch["ym"].values
        for i in range(len(ch)):
            add(node_acc, (a1[i],ym[i]), sc[i],g[i],m[i],t[i])
            add_topk(node_topk, (a1[i],ym[i]), sc[i],g[i],m[i],t[i])
            if pd.notna(a2[i]):
                add(pair_acc, (a1[i],a2[i],ym[i]), sc[i],g[i],m[i],t[i])
                add_topk(pair_topk,(a1[i],a2[i],ym[i]), sc[i],g[i],m[i],t[i])
        if b % 10 == 0: print(f"  row-group {b}/{pf.num_row_groups}")
    return node_acc, pair_acc, node_topk, pair_topk

node_acc, pair_acc, node_topk, pair_topk = stream_score()
print("done streaming.")

  row-group 0/8
done streaming.


In [10]:
def _finalize_node(acc, topk, out, use_topk, year_lo=2017, year_hi=2024):
    rows=[]
    for key,val in (topk.items() if use_topk else acc.items()):
        iso, ym = key
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            if not buf: continue
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf); tsum=sum(x[3] for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((iso, ym.year, n, ssum/n, smax, gm/max(msum,1), tsum/n))
    df = pd.DataFrame(rows, columns=["iso3","year","n_events","score_mean","score_max","goldstein_wmean","tone_mean"])
    ann = df.groupby(["iso3","year"]).agg(
        events_total=("n_events","sum"), score_mean=("score_mean","mean"),
        score_max=("score_max","max"), score_vol=("score_mean","std"),
        goldstein_wmean=("goldstein_wmean","mean"), tone_mean=("tone_mean","mean"),
        active_months=("n_events", lambda s:(s>0).sum())).reset_index()
    ann = ann[ann["year"].between(year_lo, year_hi)]
    ann["reporterCode"]=ann["iso3"].map(iso2m49); ann=ann.dropna(subset=["reporterCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann.to_parquet(out,index=False); print("saved",out,ann.shape)

_finalize_node(node_acc, node_topk, os.path.join(INTERIM_DIR, "gdelt_features_by_country_year_2017_2024.parquet"), use_topk=False)
_finalize_node(node_acc, node_topk, os.path.join(INTERIM_DIR, "gdelt_features_topk_2017_2024.parquet"),            use_topk=True)


def _finalize_pair(acc, topk, out, use_topk, year_lo=2017, year_hi=2024):
    rows=[]
    items = topk.items() if use_topk else acc.items()
    for key, val in items:
        a1, a2, ym = key
        if ym.year < year_lo or ym.year > year_hi:
            continue
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            if not buf: continue
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((a1, a2, ym.year, n, ssum/n, smax, gm/max(msum,1)))
    df = pd.DataFrame(rows, columns=["Actor1CountryCode","Actor2CountryCode","year",
                                     "pair_events","pair_score_mean","pair_score_max","pair_gold_mean"])
    ann = df.groupby(["Actor1CountryCode","Actor2CountryCode","year"]).agg(
        pair_events=("pair_events","sum"),
        pair_score_mean=("pair_score_mean","mean"),
        pair_score_max=("pair_score_max","max"),
        pair_gold_mean=("pair_gold_mean","mean")).reset_index()
    ann["reporterCode"]=ann["Actor1CountryCode"].map(iso2m49)
    ann["partnerCode"] =ann["Actor2CountryCode"].map(iso2m49)
    ann=ann.dropna(subset=["reporterCode","partnerCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann["partnerCode"] =ann["partnerCode"].astype(int)
    ann.to_parquet(out, index=False); print("saved", out, ann.shape)

_finalize_pair(pair_acc, pair_topk, os.path.join(INTERIM_DIR, "gdelt_bilateral_by_pair_year_2017_2024.parquet"), use_topk=False)
_finalize_pair(pair_acc, pair_topk, os.path.join(INTERIM_DIR, "gdelt_bilateral_topk_2017_2024.parquet"),         use_topk=True)

saved ../../data/interim/gdelt_features_by_country_year_2017_2024.parquet (344, 10)
saved ../../data/interim/gdelt_features_topk_2017_2024.parquet (344, 10)
saved ../../data/interim/gdelt_bilateral_by_pair_year_2017_2024.parquet (5570, 9)
saved ../../data/interim/gdelt_bilateral_topk_2017_2024.parquet (5570, 9)


## Final check

Confirms 2024 actually made it into all four output files, same style of sanity check as
the original notebook's post-pipeline cells.

In [11]:
for f in [os.path.join(INTERIM_DIR, "gdelt_features_by_country_year_2017_2024.parquet"),
          os.path.join(INTERIM_DIR, "gdelt_features_topk_2017_2024.parquet"),
          os.path.join(INTERIM_DIR, "gdelt_bilateral_by_pair_year_2017_2024.parquet"),
          os.path.join(INTERIM_DIR, "gdelt_bilateral_topk_2017_2024.parquet")]:
    d = pd.read_parquet(f)
    print(f"{f}: {d.shape}  years={sorted(d['year'].unique())}")

../../data/interim/gdelt_features_by_country_year_2017_2024.parquet: (344, 10)  years=[2023, 2024]
../../data/interim/gdelt_features_topk_2017_2024.parquet: (344, 10)  years=[2023, 2024]
../../data/interim/gdelt_bilateral_by_pair_year_2017_2024.parquet: (5570, 9)  years=[2023, 2024]
../../data/interim/gdelt_bilateral_topk_2017_2024.parquet: (5570, 9)  years=[2023, 2024]


**After this runs**, point `train_benchmark.py` / `test_bilateral_pairs.py` / any retraining
driver at `all_products_ready_2017_2024.parquet` (from the Comtrade extraction notebook) and
these four `*_2017_2024.parquet` GDELT files, then extend `load_or_split()`'s cutoff so
train=2017-2023 and the new held-out test year is 2024, instead of the current 2017-2022/2023
split. That retraining step still has to run on your machine (GPU/DGL, real data) --
nothing here trains anything, it only produces the extended input files.